# Program 1: Text Preprocessing using NLTK and spaCy
**Course:** CS5202 – Natural Language Processing Laboratory  
**Dataset:** UCI SMS Spam Collection

## Aim
Build and evaluate a text preprocessing pipeline that performs normalization, tokenization, stop-word removal, stemming, lemmatization, part-of-speech (POS) tagging, and named entity recognition (NER).

## Workflow
1. Load and inspect the SMS dataset.
2. Clean the complete dataset and report cleaning statistics.
3. Draw a stratified sample of up to 500 messages for library comparison.
4. Compare NLTK and spaCy tokenization and lemmatization.
5. Generate POS summaries and inspect named entities.
6. Review sample outputs, errors, limitations, and ethical considerations.

**Privacy note:** SMS messages may contain personal information. Avoid publishing raw messages or identifiable outputs in a public repository.

## 1. Install dependencies
Run this cell in Google Colab or a Jupyter environment.

In [5]:
%pip -q install ucimlrepo nltk spacy pandas scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Import libraries and download NLTK resources

In [6]:
import re
import html
import random
import warnings
from collections import Counter

import pandas as pd
import nltk
import spacy
from ucimlrepo import fetch_ucirepo
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"Could not download {resource}: {exc}")

nlp = spacy.load("en_core_web_sm")
nltk_stop_words = set(stopwords.words("english"))
spacy_stop_words = nlp.Defaults.stop_words
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("NLTK version:", nltk.__version__)
print("spaCy version:", spacy.__version__)
print("spaCy pipeline:", nlp.pipe_names)

NLTK version: 3.9.1
spaCy version: 3.8.16
spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 3. Load and inspect the UCI SMS Spam Collection

In [7]:
import pandas as pd
import urllib.request
import zipfile
import io

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

with urllib.request.urlopen(url) as response:
    zip_data = response.read()

with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    filename = next(
        n for n in z.namelist()
        if n.endswith("SMSSpamCollection")
    )
    df = pd.read_csv(
        z.open(filename),
        sep="\t",
        header=None,
        names=["label", "message"],
        encoding="utf-8"
    )

print("Dataset shape:", df.shape)
print(df.head())
print(df["label"].value_counts())

Dataset shape: (5572, 2)
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64


## 4. Inspect text characteristics without exposing messages

In [8]:
raw = df["message"].fillna("")
inspection = pd.Series({
    "documents": len(df),
    "missing_messages": int(df["message"].isna().sum()),
    "messages_with_HTML_tags": int(raw.str.contains(r"<[^>]+>", regex=True).sum()),
    "messages_with_URLs": int(raw.str.contains(r"https?://|www\\.", case=False, regex=True).sum()),
    "messages_with_non-ASCII_characters": int(raw.str.contains(r"[^\\x00-\\x7F]", regex=True).sum()),
    "messages_with_repeated_whitespace": int(raw.str.contains(r"\\s{2,}", regex=True).sum()),
    "messages_with_punctuation": int(raw.str.contains(r"[!?,.;:]", regex=True).sum()),
    "average_characters_per_message": round(raw.str.len().mean(), 2),
})
display(inspection.to_frame("value"))

,value
documents,5572.00
missing_messages,0.00
messages_with_HTML_tags,6.00
messages_with_URLs,20.00
messages_with_non-ASCII_characters,5570.00
messages_with_repeated_whitespace,0.00
messages_with_punctuation,4933.00
average_characters_per_message,80.49


## 5. Normalize and clean messages
This keeps meaningful words and punctuation, replaces URLs with a marker, unescapes HTML entities, and collapses repeated whitespace.

In [9]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)                    # remove HTML tags
    text = re.sub(r"https?://\\S+|www\\.\\S+", " URL ", text, flags=re.I)
    text = text.lower()
    text = re.sub(r"[^\\w\\s'!?.,:;/-]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].map(clean_text)
df["raw_characters"] = df["message"].fillna("").str.len()
df["clean_characters"] = df["clean_message"].str.len()
df["raw_word_count"] = df["message"].fillna("").str.findall(r"\\b\\w+\\b").str.len()
df["clean_word_count"] = df["clean_message"].str.findall(r"\\b\\w+\\b").str.len()

cleaning_summary = pd.DataFrame({
    "metric": ["messages", "mean raw characters", "mean cleaned characters",
               "mean raw word count", "mean cleaned word count",
               "empty after cleaning"],
    "value": [len(df), round(df["raw_characters"].mean(), 2),
              round(df["clean_characters"].mean(), 2),
              round(df["raw_word_count"].mean(), 2),
              round(df["clean_word_count"].mean(), 2),
              int((df["clean_message"] == "").sum())]
})
display(cleaning_summary)

,metric,value
0,messages,5572.00
1,mean raw characters,80.49
2,mean cleaned characters,67.56
3,mean raw word count,0.00
4,mean cleaned word count,0.00
5,empty after cleaning,56.00


## 6. Create a stratified sample of up to 500 messages
The full dataset is used for cleaning statistics; detailed NLTK–spaCy comparison uses a reproducible stratified sample.

In [10]:
sample_size = min(500, len(df))
if df["label"].nunique() > 1 and sample_size >= df["label"].nunique():
    comparison_df, _ = train_test_split(
        df, train_size=sample_size, random_state=SEED,
        stratify=df["label"]
    )
else:
    comparison_df = df.sample(n=sample_size, random_state=SEED)

comparison_df = comparison_df.copy().reset_index(drop=True)
print("Comparison sample size:", len(comparison_df))
display(comparison_df["label"].value_counts().rename_axis("label").to_frame("sample_count"))

Comparison sample size: 500


,sample_count
label,
ham,433
spam,67


## 7. NLTK tokenization, stop-word removal, stemming, and lemmatization

In [11]:
def nltk_preprocess(text):
    tokens = word_tokenize(text)
    word_tokens = [t for t in tokens if re.search(r"\\w", t)]
    filtered = [t for t in word_tokens if t.lower() not in nltk_stop_words]
    stems = [stemmer.stem(t.lower()) for t in filtered]
    lemmas = [lemmatizer.lemmatize(t.lower()) for t in filtered]
    return {
        "nltk_tokens": word_tokens,
        "nltk_token_count": len(word_tokens),
        "nltk_filtered_tokens": filtered,
        "nltk_filtered_count": len(filtered),
        "nltk_stems": stems,
        "nltk_lemmas": lemmas,
    }

nltk_results = comparison_df["clean_message"].map(nltk_preprocess).apply(pd.Series)
comparison_df = pd.concat([comparison_df, nltk_results], axis=1)
display(comparison_df[["nltk_token_count", "nltk_filtered_count"]].describe())

,nltk_token_count,nltk_filtered_count
count,500.0,500.0
mean,0.0,0.0
std,0.0,0.0
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,0.0,0.0


## 8. spaCy tokenization, stop-word removal, lemmatization, POS tagging, and NER

In [12]:
def spacy_preprocess(text):
    doc = nlp(text)
    lexical_tokens = [t for t in doc if not t.is_space and not t.is_punct]
    filtered = [t for t in lexical_tokens if not t.is_stop and t.is_alpha]
    pos_counts = Counter(t.pos_ for t in lexical_tokens)
    entities = [{"text": ent.text, "label": ent.label_} for ent in doc.ents]
    return {
        "spacy_tokens": [t.text for t in lexical_tokens],
        "spacy_token_count": len(lexical_tokens),
        "spacy_filtered_tokens": [t.text for t in filtered],
        "spacy_filtered_count": len(filtered),
        "spacy_lemmas": [t.lemma_.lower() for t in filtered],
        "spacy_pos": [(t.text, t.pos_, t.tag_) for t in lexical_tokens],
        "spacy_pos_counts": dict(pos_counts),
        "spacy_entities": entities,
    }

spacy_results = comparison_df["clean_message"].map(spacy_preprocess).apply(pd.Series)
comparison_df = pd.concat([comparison_df, spacy_results], axis=1)
display(comparison_df[["spacy_token_count", "spacy_filtered_count"]].describe())

,spacy_token_count,spacy_filtered_count
count,500.000000,500.000000
mean,4.618000,4.380000
std,3.758234,3.576756
min,0.000000,0.000000
25%,2.000000,2.000000
50%,4.000000,3.000000
75%,6.000000,6.000000
max,25.000000,23.000000


## 9. Compare NLTK and spaCy on five examples
SMS abbreviations, spelling variants, and short fragments can make tokenization, lemmatization, POS tagging, and NER difficult.

In [13]:
comparison_columns = [
    "clean_message", "nltk_tokens", "spacy_tokens",
    "nltk_lemmas", "spacy_lemmas", "spacy_pos", "spacy_entities"
]
display(comparison_df[comparison_columns].head(5).to_string(index=False))

"                                                                                                         clean_message nltk_tokens        spacy_tokens nltk_lemmas       spacy_lemmas                                                                                       spacy_pos                                                                              spacy_entities\ns              s           '                           .               s        s           w                       s.          [] [s, s, s, s, w, s.]          []    [s, s, s, s, w] [(s, PROPN, NNP), (s, PART, POS), (s, PART, POS), (s, PART, POS), (w, X, FW), (s., PROPN, NNP)]                                                                                          []\n                                                                          w      s  s        s                       .          []        [w, s, s, s]          []       [w, s, s, s]                                   [(w, PROPN, NNP), (s, PART, POS), (s, PAR

## 10. Aggregate token and POS summaries

In [14]:
token_count_summary = pd.DataFrame({
    "library": ["NLTK", "spaCy"],
    "mean_tokens_before_stopword_removal": [
        comparison_df["nltk_token_count"].mean(),
        comparison_df["spacy_token_count"].mean()
    ],
    "mean_tokens_after_stopword_removal": [
        comparison_df["nltk_filtered_count"].mean(),
        comparison_df["spacy_filtered_count"].mean()
    ]
}).round(2)
display(token_count_summary)

all_pos = Counter()
for counts in comparison_df["spacy_pos_counts"]:
    all_pos.update(counts)
pos_summary = pd.DataFrame(all_pos.most_common(), columns=["POS tag", "count"])
display(pos_summary)

all_entities = Counter()
for entities in comparison_df["spacy_entities"]:
    for entity in entities:
        all_entities[entity["label"]] += 1
entity_summary = pd.DataFrame(all_entities.most_common(), columns=["entity label", "count"])
display(entity_summary if not entity_summary.empty else pd.DataFrame(columns=["entity label", "count"]))

,library,mean_tokens_before_stopword_removal,mean_tokens_after_stopword_removal
0,NLTK,0.00,0.00
1,spaCy,4.62,4.38


,POS tag,count
0,PROPN,811
1,PART,670
2,VERB,268
3,X,254
4,NOUN,253
5,INTJ,18
6,PRON,14
7,AUX,13
8,NUM,4
9,PUNCT,2


,entity label,count
0,PERSON,127
1,ORG,19
2,PRODUCT,9
3,CARDINAL,7
4,QUANTITY,2
5,GPE,2
6,NORP,1


## 11. Error inspection and limitations

In [15]:
# Select a few records with non-empty text for manual inspection.
# Do not print or commit this table publicly if messages contain personal information.
error_review = comparison_df.loc[
    comparison_df["clean_message"].str.len() > 0,
    ["label", "clean_message", "nltk_lemmas", "spacy_lemmas", "spacy_entities"]
].head(10)
display(error_review)

,label,clean_message,nltk_lemmas,spacy_lemmas,spacy_entities
0,ham,s s ' ...,[],"[s, s, s, s, w]",[]
1,ham,w s s s .,[],"[w, s, s, s]",[]
2,ham,w s w s ' ...,[],"[w, s, w, s, s, s]",[{'text': 'w s w ...
3,ham,. w s . .,[],"[w, s]",[]
4,ham,. s -- s ' ...,[],"[s, s, s]",[]
5,spam,s w s ...,[],"[s, w, s, ss, w, s]",[{'text': 's w ...
6,ham,. s s ...,[],"[s, s, s, s, w, w, s]",[]
7,spam,'s ! w s ...,[],"[w, s, s, s, w, s, s, s, s, s]",[]
8,ham,s s ?,[],"[s, s]",[]
9,ham,.. ? s,[],[s],[]


### Discussion prompts (complete after running)
- **Tokenization:** Did the libraries split contractions, punctuation, and SMS shorthand differently?
- **Stop words:** Stop-word removal can discard useful meaning in some tasks; it should be chosen for the downstream objective.
- **Stemming vs. lemmatization:** Porter stemming applies heuristic suffix rules and may produce non-words; WordNet lemmatization is dictionary-based but may need POS information. spaCy provides contextual token annotations and lemmas.
- **POS tagging:** Short, ungrammatical SMS messages and abbreviations can be ambiguous.
- **NER:** SMS messages often contain few conventional named entities. URLs, phone numbers, abbreviations, and personal names may be missed or mislabeled.
- **Limitations:** Results depend on the spaCy model, NLTK resources, preprocessing choices, and dataset composition. Lowercasing and symbol removal may discard useful signals.
- **Ethics:** Do not publish raw SMS content; inspect examples privately and share only aggregated or suitably anonymized results.

## 12. Save reproducible summary files
This saves aggregate statistics rather than exporting raw messages.

In [18]:
os.makedirs("results", exist_ok=True)
cleaning_summary.to_csv("results/cleaning_summary.csv", index=False)
token_count_summary.to_csv("results/token_count_comparison.csv", index=False)
pos_summary.to_csv("results/spacy_pos_summary.csv", index=False)
entity_summary.to_csv("results/spacy_entity_summary.csv", index=False)
print("Saved aggregate result tables in ./results/")

Saved aggregate result tables in ./results/


In [17]:
import os